# Non-invasive glucose prediction - exploratory analysis and regression models

This notebook supports the project scope: **physiological sensor–derived features → supervised regression for glucose level**, with **data quality checks**, **exploratory analysis**, and **model development and evaluation**.

**Important:** rows are repeated per `Patient_Id` (within-subject time/sample index). Models are evaluated with a **patient-level holdout** so that the same individual does not appear in both training and test sets (reduces optimistic bias from leakage).

## Environment

Run once if packages are missing:

In [1]:
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").is_dir() and (PROJECT_ROOT.parent / "dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "dataset" / "augmented_dataset_new.csv"
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)



## 1. Load data

Expected columns: demographics, anthropometrics, PPG-derived features, and `Glucose_level` as the regression target.

In [ ]:
candidates = [
    DATA_PATH,
    PROJECT_ROOT / "dataset" / "augmented_dataset.csv",
    Path.cwd() / "dataset" / "augmented_dataset_new.csv",
    Path.home() / "OneDrive" / "Desktop" / "glucose-level-prediction" / "dataset" / "augmented_dataset_new.csv",
]

data_file = next((p for p in candidates if p.exists()), None)
if data_file is None:
    print("Tried:", *candidates, sep="\n  ")
    raise FileNotFoundError("Put CSV under dataset/ or fix DATA_PATH.")

df = pd.read_csv(data_file, low_memory=False)
print("Shape:", df.shape)
df.head()



File not found at: C:\Users\555555\augmented_dataset.csv
Current working directory: C:\Users\555555
Files in current directory: [WindowsPath('C:/Users/555555/Anambra_Basin_Benue_Trough_Well_Data.csv'), WindowsPath('C:/Users/555555/backtest_summary.csv'), WindowsPath('C:/Users/555555/behavioral_dataset.csv'), WindowsPath('C:/Users/555555/BTCUSDT_1h_2021.csv'), WindowsPath('C:/Users/555555/btc_binance_1h_1y.csv'), WindowsPath('C:/Users/555555/BTC_CG_recent.csv'), WindowsPath('C:/Users/555555/car-sales (2).csv'), WindowsPath('C:/Users/555555/car-sales-missing-data.csv'), WindowsPath('C:/Users/555555/car-sales.csv'), WindowsPath('C:/Users/555555/carbon.csv'), WindowsPath('C:/Users/555555/CICDDoS.csv'), WindowsPath('C:/Users/555555/clean-dataset.csv'), WindowsPath('C:/Users/555555/coingecko_ethbtc_ohlc.csv'), WindowsPath('C:/Users/555555/coingecko_ethbtc_ohlc_volume.csv'), WindowsPath('C:/Users/555555/coingecko_ethbtc_ohlc_volume_indicators.csv'), WindowsPath('C:/Users/555555/correlation_su

FileNotFoundError: Place augmented_dataset.csv in the same folder as this notebook.

## 2. Data dictionary and types

In [ ]:
dictionary = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "n_unique": df.nunique().values,
    }
)
dictionary

## 3. Data quality — missing values, duplicates, basic sanity

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(3)
qc = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
qc = qc[qc["missing_count"] > 0].sort_values("missing_count", ascending=False)
print("Columns with any missing values:")
display(qc if len(qc) else "None")

dup_rows = df.duplicated().sum()
print(f"Fully duplicate rows: {dup_rows:,}")

if {"pl", "index"}.issubset(df.columns):
    same = (df["pl"] == df["index"]).mean()
    print(f"Share of rows where pl == index: {same:.4f}")

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", numeric_cols)
df[numeric_cols].describe().T

## 4. Target variable — `Glucose_level`

Distribution, outliers (Tukey IQR rule), and normality check on a random subsample if the full series is heavy to plot.

In [ ]:
y = df["Glucose_level"]
q1, q3 = y.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_mask = (y < low) | (y > high)
print(y.describe())
print(f"IQR outliers (rows): {outlier_mask.sum():,} ({outlier_mask.mean()*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Glucose_level distribution")
sns.boxplot(x=y, ax=axes[1], color="lightblue")
axes[1].set_title("Glucose_level boxplot")
plt.tight_layout()
plt.show()

sub = y.sample(min(50_000, len(y)), random_state=RANDOM_STATE)
stat, p = stats.normaltest(sub)
print(f"D'Agostino–Pearson normality test on n={len(sub):,} subsample: statistic={stat:.3f}, p-value={p:.2e}")

## 5. Patient structure

How many rows per patient informs whether within-patient correlation is strong and why grouped splitting matters.

In [ ]:
if "Patient_Id" not in df.columns:
    raise ValueError("Expected column Patient_Id for grouped evaluation.")

per_patient = df.groupby("Patient_Id").size().rename("n_rows")
print(per_patient.describe())

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(per_patient, bins=40, kde=True, ax=ax, color="teal")
ax.set_title("Rows per Patient_Id")
plt.show()

## 6. Feature–target relationships

Pearson correlation with `Glucose_level` for numeric inputs (excluding identifiers and target).

In [ ]:
exclude_corr = {"Patient_Id", "Glucose_level"}
feat_for_corr = [c for c in numeric_cols if c not in exclude_corr]
corr_series = df[feat_for_corr + ["Glucose_level"]].corr()["Glucose_level"].drop("Glucose_level").sort_values(key=abs, ascending=False)
print(corr_series)

plt.figure(figsize=(8, max(4, 0.35 * len(corr_series))))
corr_series.sort_values().plot(kind="barh", color="slategray")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Pearson correlation with Glucose_level")
plt.tight_layout()
plt.show()

In [ ]:
top_k = min(5, len(feat_for_corr))
top_feats = corr_series.abs().sort_values(ascending=False).head(top_k).index.tolist()

sample = df.sample(min(8_000, len(df)), random_state=RANDOM_STATE)
pair_cols = top_feats + ["Glucose_level"]
sns.pairplot(sample[pair_cols], corner=True, plot_kws={"s": 8, "alpha": 0.35})
plt.suptitle("Pairplot (random subsample)", y=1.02)
plt.show()

In [ ]:
cm = df[feat_for_corr + ["Glucose_level"]].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(cm, cmap="vlag", center=0, annot=False, linewidths=0.2)
plt.title("Correlation matrix (numeric features + target)")
plt.tight_layout()
plt.show()

## 7. End-to-end ML pipeline (overview)

1. **Feature engineering (Section 7a)** — row-wise physiology-derived columns (no future information); redundant raw columns dropped where they duplicate engineered signals.
2. **Train / test split** - `GroupShuffleSplit` on `Patient_Id` so all rows from a patient stay in one split.
3. **Preprocessing (inside sklearn `Pipeline`, fit on train only)**  
   - **Numeric:** `SimpleImputer` (median) → **`StandardScaler`** (zero mean / unit variance) for `Ridge (Standard)`; same imputer → **`RobustScaler`** (median/IQR–based, heavy-tail–friendly) for `Ridge (Robust)`.  
   - **Low-cardinality categoricals:** `SimpleImputer` (most frequent) → **`OneHotEncoder`** (`drop='if_binary'`) for `Gender` when present.  
   - **Tree models:** median imputation only (no scaling; splits are scale-invariant).
4. **Estimators** - Dummy baseline, Ridge (two scaling choices), Random Forest, Histogram Gradient Boosting.
5. **Evaluation** - MAE, RMSE, R² on the held-out **patient group**.

This matches a standard supervised regression workflow: **imputation → encoding/scaling → model**, all in one `Pipeline` per variant so test data never sees training statistics before `transform`.

### 7a — Feature engineering (row-wise, no leakage)

Derived columns use only values from the **same row** (same time/sample as the PPG window). They are standard PPG / hemodynamics–style summaries: pulse pressure–like separation of systolic vs diastolic peaks, simple stiffness / augmentation proxies, and intensity normalized by heart rate.

In [ ]:
def add_engineered_features(raw: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with extra numeric columns for modeling."""
    d = raw.copy()

    if {"Systolic_Peak", "Diastolic_Peak"}.issubset(d.columns):
        d["Pulse_Pressure_Peak"] = d["Systolic_Peak"] - d["Diastolic_Peak"]
        sys_safe = d["Systolic_Peak"].replace(0, np.nan)
        dia_safe = d["Diastolic_Peak"].replace(0, np.nan)
        d["Peak_Sys_Dia_Ratio"] = d["Systolic_Peak"] / dia_safe
        d["Augmentation_Index_Proxy"] = (d["Systolic_Peak"] - d["Diastolic_Peak"]) / sys_safe

    if {"PPG_Signal", "Heart_Rate"}.issubset(d.columns):
        hr = d["Heart_Rate"].replace(0, np.nan)
        d["PPG_per_HeartRate"] = d["PPG_Signal"] / hr

    if {"Pulse_Area", "PPG_Signal"}.issubset(d.columns):
        ppg = d["PPG_Signal"].replace(0, np.nan)
        d["PulseArea_per_PPG"] = d["Pulse_Area"] / ppg

    if "Pulse_Area" in d.columns:
        d["log1p_Pulse_Area"] = np.log1p(d["Pulse_Area"].clip(lower=0))

    new_cols = [c for c in d.columns if c not in raw.columns]
    for c in new_cols:
        d[c] = d[c].replace([np.inf, -np.inf], np.nan)

    return d


df_model = add_engineered_features(df)
print("Added columns:", [c for c in df_model.columns if c not in df.columns])
df_model[[c for c in df_model.columns if c not in df.columns]].describe().T.head(20)

### 7b — Design matrix, groups, and column roles

- **Excluded from X:** `Patient_Id`, `Glucose_level`, redundant `pl` when identical to `index`.
- **Categorical path:** `Gender` (if present, low cardinality) → one-hot after imputation.
- **Numeric path:** all other modeling columns → imputation + scaling (see next cell).

In [ ]:
TARGET = "Glucose_level"
GROUP = "Patient_Id"

drop_extra = set()
if {"pl", "index"}.issubset(df_model.columns) and (df_model["pl"] == df_model["index"]).all():
    drop_extra.add("pl")

feature_cols = [
    c
    for c in df_model.columns
    if c not in {TARGET, GROUP} | drop_extra
]

X = df_model[feature_cols]
y = df_model[TARGET].values
groups = df_model[GROUP].values

CANDIDATE_CAT = ["Gender"]
categorical_features = [
    c
    for c in CANDIDATE_CAT
    if c in X.columns and X[c].nunique() <= 12
]
numeric_features = [c for c in X.columns if c not in categorical_features]

print("Categorical (encoded):", categorical_features)
print("Numeric (impute + scale or tree-impute only):", len(numeric_features), "columns")
X.head()

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups[train_idx]

train_patients = set(np.unique(groups_train))
test_patients = set(np.unique(groups[test_idx]))
overlap = train_patients & test_patients
print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Train patients: {len(train_patients):,} | Test patients: {len(test_patients):,}")
print(f"Patient overlap between splits: {len(overlap)} (should be 0)")

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred, name="model"):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


print("sklearn version:", sklearn.__version__)

### 7c — Preprocessing + estimators (`sklearn.pipeline.Pipeline`)

Each model is a single `Pipeline`: **preprocessor** (fit on training rows only) then **regressor**. Categorical and numeric branches are combined with `ColumnTransformer`.

In [ ]:
num_pipe_std = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

num_pipe_robust = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

tree_num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                drop="if_binary",
                sparse_output=False,
                handle_unknown="ignore",
            ),
        ),
    ]
)

linear_transformers_std = [("num", num_pipe_std, numeric_features)]
linear_transformers_robust = [("num", num_pipe_robust, numeric_features)]
tree_transformers = [("num", tree_num_pipe, numeric_features)]

if categorical_features:
    linear_transformers_std.append(("cat", cat_pipe, categorical_features))
    linear_transformers_robust.append(("cat", cat_pipe, categorical_features))
    tree_transformers.append(("cat", cat_pipe, categorical_features))

preprocess_linear_std = ColumnTransformer(
    transformers=linear_transformers_std,
    remainder="drop",
)

preprocess_linear_robust = ColumnTransformer(
    transformers=linear_transformers_robust,
    remainder="drop",
)

preprocess_tree = ColumnTransformer(
    transformers=tree_transformers,
    remainder="drop",
)

models = {
    "Dummy (mean)": Pipeline(
        [("prep", preprocess_linear_std), ("model", DummyRegressor(strategy="mean"))]
    ),
    "Ridge (StandardScaler)": Pipeline(
        [("prep", preprocess_linear_std), ("model", Ridge(alpha=1.0))]
    ),
    "Ridge (RobustScaler)": Pipeline(
        [("prep", preprocess_linear_robust), ("model", Ridge(alpha=1.0))]
    ),
    "RandomForest": Pipeline(
        [
            ("prep", preprocess_tree),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=100,
                    max_depth=16,
                    min_samples_leaf=20,
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "HistGradientBoosting": Pipeline(
        [
            ("prep", preprocess_tree),
            (
                "model",
                HistGradientBoostingRegressor(
                    max_depth=8,
                    learning_rate=0.05,
                    max_iter=200,
                    min_samples_leaf=40,
                    l2_regularization=0.1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

In [ ]:
results = []
fitted = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append(regression_metrics(y_test, pred, name=name))
    fitted[name] = pipe

metrics_df = pd.DataFrame(results).set_index("model").sort_values("MAE")
metrics_df

In [ ]:
best_name = metrics_df["MAE"].idxmin()
best = fitted[best_name]
print("Best model by test MAE:", best_name)

pred_best = best.predict(X_test)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred_best, alpha=0.15, s=10, edgecolors="none")
lims = [min(y_test.min(), pred_best.min()), max(y_test.max(), pred_best.max())]
ax.plot(lims, lims, "r--", lw=1, label="ideal")
ax.set_xlabel("Actual glucose")
ax.set_ylabel("Predicted glucose")
ax.set_title(f"Test set: actual vs predicted ({best_name})")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

residuals = y_test - pred_best
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(residuals, kde=True, ax=axes[0], color="coral")
axes[0].set_title("Residual distribution")
axes[1].scatter(pred_best, residuals, alpha=0.12, s=8)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs fitted")
plt.tight_layout()
plt.show()

## 8. Feature importance 

For `RandomForest`, importances are computed from the fitted pipeline’s last step.

In [ ]:
if "RandomForest" in fitted:
    rf_pipe = fitted["RandomForest"]
    prep = rf_pipe.named_steps["prep"]
    rf = rf_pipe.named_steps["model"]
    feat_names = prep.get_feature_names_out()
    imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=True)
    plt.figure(figsize=(8, max(4, 0.35 * len(imp))))
    imp.plot(kind="barh", color="darkgreen")
    plt.title("RandomForest feature importances")
    plt.tight_layout()
    plt.show()
else:
    print("RandomForest not in fitted models.")

## 9. grouped cross-validation (stability)

Set `RUN_GROUP_CV = True` in the next cell to run 3-fold **GroupKFold** (same patients never appear in different folds). Left **False** by default because it retrains on most of the data three times and can take a while on large files.

In [ ]:
RUN_GROUP_CV = False
if RUN_GROUP_CV:
    gkf = GroupKFold(n_splits=3)
    print("Running grouped CV (this may take several minutes)...")
    hgb_full = Pipeline(
        [
            ("prep", preprocess_tree),
            (
                "model",
                HistGradientBoostingRegressor(
                    max_depth=8,
                    learning_rate=0.05,
                    max_iter=150,
                    min_samples_leaf=50,
                    l2_regularization=0.1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    scores = cross_val_score(
        hgb_full,
        X,
        y,
        cv=gkf,
        groups=groups,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    )
    print("HistGradientBoosting 3-fold GroupKFold MAE:", (-scores).round(4))
    print("Mean MAE:", float(-scores.mean()), "std:", float(scores.std()))
else:
    print("Skipped grouped CV (set RUN_GROUP_CV = True to enable).")

In [ ]:
import joblib
from sklearn.model_selection import GridSearchCV

# Save the best model
joblib.dump(best, MODEL_DIR / "best_glucose_model.pkl")
print(f"Model '{best_name}' saved as 'best_glucose_model.pkl'")

# Hyperparameter tuning for the best model using GridSearchCV

if best_name == "HistGradientBoosting":
    param_grid = {
        "model__max_depth": [6, 8, 10],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_iter": [150, 200, 250],
        "model__l2_regularization": [0.05, 0.1, 0.2],
    }
elif best_name == "RandomForest":
    param_grid = {
        "model__max_depth": [12, 16, 20],
        "model__min_samples_leaf": [10, 20, 30],
        "model__n_estimators": [100, 150, 200],
    }
elif best_name == "Ridge (StandardScaler)":
    param_grid = {"model__alpha": [0.1, 1.0, 10.0, 100.0]}
else:
    param_grid = {}

if param_grid:
    print(f"Tuning hyperparameters for {best_name}...")
    grid_search = GridSearchCV(
        best,
        param_grid,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        verbose=1,
    )
    grid_search.fit(X_train, y_train)
    print(f"Best params: {grid_search.best_params_}")
    print(f"Best CV MAE: {-grid_search.best_score_:.4f}")
    
    best_tuned = grid_search.best_estimator_
    pred_tuned = best_tuned.predict(X_test)
    tuned_metrics = regression_metrics(y_test, pred_tuned, name=f"{best_name} (tuned)")
    print("Test metrics (tuned):", tuned_metrics)
    
    joblib.dump(best_tuned, MODEL_DIR / "best_glucose_model_tuned.pkl")
    print("Tuned model saved as 'best_glucose_model_tuned.pkl'")
else:
    print(f"No hyperparameter grid defined for {best_name}")
